In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# path setup
try:
    PROJECT_ROOT = Path(__file__).parent.parent
except NameError:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

data_processed = PROJECT_ROOT / "data" / "processed"

# load FINAL dataset (01c)
df = pd.read_parquet(data_processed / "makassar_final_dataset.parquet")

print("DATA SHAPE:", df.shape)
df.head()

DATA SHAPE: (4922658, 12)


,grid_id,date,rainfall_real,rain_real_lag_1,rain_real_lag_3,rain_real_lag_7,rain_real_lag_14,rain_real_7d_sum,rain_real_30d_sum,month,spatial_factor,flood_label_v2
0,227,2018-03-31,26.024028,16.232391,0.000000,8.374993,1.300432,98.321737,510.811864,3,0.949816,0
1,227,2018-04-01,20.123071,26.024028,20.082396,5.076677,1.461669,113.368131,524.406615,4,0.949816,0
2,227,2018-04-02,29.019689,20.123071,16.232391,25.325795,11.423366,117.062025,542.905413,4,0.949816,0
3,227,2018-04-03,8.802669,29.019689,26.024028,5.580451,0.000000,120.284244,534.535854,4,0.949816,1
4,227,2018-04-04,13.563821,8.802669,20.123071,0.000000,9.624444,133.848065,519.164902,4,0.949816,0


In [2]:
# pastikan pakai label baru
df = df.drop(columns=["flood_label"], errors="ignore")

# sort
df = df.sort_values(["grid_id", "date"])

# cek label
print("Flood ratio:", df["flood_label_v2"].mean())

Flood ratio: 0.23568243010178647


In [3]:
# split berdasarkan waktu
train_df = df[df["date"] < "2021-01-01"]
test_df  = df[df["date"] >= "2021-01-01"]

print("TRAIN:", train_df.shape)
print("TEST :", test_df.shape)

TRAIN: (2853838, 12)
TEST : (2068820, 12)


In [4]:
X_train = train_df[["grid_id"]]
y_train = train_df["flood_label_v2"]

X_test = test_df[["grid_id"]]
y_test = test_df["flood_label_v2"]

model_spatial = RandomForestClassifier(
    n_estimators=50,
    n_jobs=-1,
    random_state=42
)

model_spatial.fit(X_train, y_train)

pred_spatial = model_spatial.predict_proba(X_test)[:, 1]

auc_spatial = roc_auc_score(y_test, pred_spatial)

print("AUC (SPATIAL ONLY):", auc_spatial)

AUC (SPATIAL ONLY): 0.5161494778740996


In [5]:
temporal_features = [
    "rainfall_real",
    "rain_real_lag_3",
    "rain_real_lag_7",
    "rain_real_lag_14",
    "rain_real_30d_sum",
    "month"
]

X_train = train_df[temporal_features]
y_train = train_df["flood_label_v2"]

X_test = test_df[temporal_features]
y_test = test_df["flood_label_v2"]

model_temporal = RandomForestClassifier(
    n_estimators=100,
    n_jobs=-1,
    random_state=42
)

model_temporal.fit(X_train, y_train)

pred_temporal = model_temporal.predict_proba(X_test)[:, 1]

auc_temporal = roc_auc_score(y_test, pred_temporal)

print("AUC (TEMPORAL):", auc_temporal)

AUC (TEMPORAL): 0.5184184918894936


In [ ]:
full_features = [
    "grid_id",
    "rainfall_real",
    "rain_real_lag_3",
    "rain_real_lag_7",
    "rain_real_lag_14",
    "rain_real_30d_sum",
    "month",
    "spatial_factor"
]

X_train = train_df[full_features]
y_train = train_df["flood_label_v2"]

X_test = test_df[full_features]
y_test = test_df["flood_label_v2"]

model_full = RandomForestClassifier(
    n_estimators=100,
    n_jobs=-1,
    random_state=42
)

model_full.fit(X_train, y_train)

pred_full = model_full.predict_proba(X_test)[:, 1]

auc_full = roc_auc_score(y_test, pred_full)

print("AUC (FULL):", auc_full)

In [ ]:
print("\n=== FINAL RESULT ===")
print(f"AUC Spatial  : {auc_spatial:.4f}")
print(f"AUC Temporal : {auc_temporal:.4f}")
print(f"AUC Full     : {auc_full:.4f}")